# Data Analysis Laboratory 02: Graz Weather Data
Fink Lucas, ESE26


## Retrieving weather data for Graz

https://climatephysics.uni-graz.at/de/meteorologische-station/

In [1]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# University of Graz main building coordinates
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 47.0768,
    "longitude": 15.4509,
    "start_date": "2000-01-01",
    "end_date": "2026-11-22",  # Update to today's date
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "timezone": "Europe/Vienna"
}
responses = openmeteo.weather_api(url, params=params)
response = responses[0]

# Process hourly data
hourly = response.Hourly()
hourly_data = {
    "timestamp": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),#.tz_convert("Europe/Vienna"),
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "precipitation": hourly.Variables(1).ValuesAsNumpy(),
    "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
}

weather_graz = pd.DataFrame(hourly_data)
print(weather_graz.head())

                  timestamp  temperature_2m  precipitation  wind_speed_10m
0 1999-12-31 23:00:00+00:00      -16.313002            0.0        5.400000
1 2000-01-01 00:00:00+00:00      -17.063002            0.0        5.400000
2 2000-01-01 01:00:00+00:00      -17.413000            0.0        5.623380
3 2000-01-01 02:00:00+00:00      -17.663000            0.0        6.130579
4 2000-01-01 03:00:00+00:00      -17.763000            0.0        5.860375


In [2]:
weather_graz.to_parquet("data/weather_graz.parquet", index=False)

## Or import from data

In [3]:
import pandas as pd

weather_graz_reloaded = pd.read_parquet("data/weather_graz.parquet")

## Assignment 1

### familiarize yourself with the data

Create a few plots to get an overview over the data, maybe also analyze the DataFrame description

## climograph

A climograph consists of temperature and precepitation collected over longer periods of time for a specific location

### create a climograph for Graz

It should contain average minimum and maximum temperatures as well as cummulative precipitation and number of rainy days.
try to make use of:
- groupby
- dt
- sum
- mean
- (agg)

How does it compare to diagrams available online? On which timespan is the diagram based on?